In [5]:
import tensorflow as tf
from transformers import TFAutoModelForSequenceClassification, AutoTokenizer
from sklearn.model_selection import train_test_split
import pandas as pd

# 1. Load a Pre-trained Model
model_name = "distilbert-base-uncased"  # Or try "bert-base-uncased", "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = TFAutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 2. Prepare the Dataset
data = {
    'text': [
        "This movie is great!",
        "The film was terrible.",
        "I loved it!",
        "Highly recommend",
        "It's awful."
    ],
    'label': [1, 0, 1, 1, 0]  # 1: positive, 0: negative
}
df = pd.DataFrame(data)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'].tolist(), df['label'].tolist(), test_size=0.2
)

# Tokenize data
train_encodings = tokenizer(train_texts, truncation=True, padding=True, return_tensors="tf")
test_encodings = tokenizer(test_texts, truncation=True, padding=True, return_tensors="tf")

# Convert labels to tensors
train_labels_tf = tf.convert_to_tensor(train_labels)
test_labels_tf = tf.convert_to_tensor(test_labels)

# Create datasets
train_dataset = tf.data.Dataset.from_tensor_slices((dict(train_encodings), train_labels_tf)).batch(2) # Reduced batch size
test_dataset = tf.data.Dataset.from_tensor_slices((dict(test_encodings), test_labels_tf)).batch(2)

# 3. Fine-tune the Model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy']) # simplified optimizer and loss
model.fit(train_dataset, epochs=2)  # Reduced epochs

# 4. Evaluate the Model
results = model.evaluate(test_dataset)
print(results)



Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_layer_norm.weight', 'vocab_projector.bias', 'vocab_layer_norm.bias', 'vocab_transform.weight', 'vocab_transform.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 

Epoch 1/2
2/2 [==============================] - 20s 991ms/step - loss: 3.3060 - accuracy: 0.7500
Epoch 2/2
2/2 [==============================] - 3s 2s/step - loss: 0.6931 - accuracy: 0.5000


1/1 [==============================] - 2s 2s/step - loss: 0.6931 - accuracy: 1.0000
[0.6931471824645996, 1.0]
